# 05 · Unity Catalog governance

Applies the governance layer to every asset the journey produced:

* **Business metadata** — table and column comments (these also ground the Genie Agent).
* **Governed tags** — `Domain` = Healthcare, `medallion_layer`, and `data_classification` = pii on
  member-level tables (all values from the metastore's tag policies), and the governed `phi` tag on member-level columns
  (`member_id` = HIPAA `health_plan_beneficiary_number`). Synthetic data, tagged the way a
  real plan would so policies and audits can key off the tags.
* **Least-privilege grants** — the app's service principal can read the serving
  tables and only *write* to `offer_feedback`; analysts get read access to gold.
* **Audit** — prints the effective grants, tags and model aliases as evidence.

In [ ]:
%pip install mlflow --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "bhavik_demos")
dbutils.widgets.text("schema", "offer_prioritization")
dbutils.widgets.text("app_sp_client_id", "f9bec3ef-421b-4a07-bf08-46c6df20c976")
dbutils.widgets.text("analyst_group", "account users")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
FQ = f"{CATALOG}.{SCHEMA}"
APP_SP = dbutils.widgets.get("app_sp_client_id")
ANALYSTS = dbutils.widgets.get("analyst_group")

In [ ]:
import json

results = []
def run(sql):
    try:
        spark.sql(sql)
        results.append(("OK", sql))
    except Exception as e:
        results.append(("SKIP", f"{sql}  -- {str(e).splitlines()[0][:140]}"))

# Feedback table written by the app (approve / reject / comment per recommendation)
run(f"""CREATE TABLE IF NOT EXISTS {FQ}.offer_feedback (
          member_id STRING, offer_id STRING, feedback STRING,
          feedback_text STRING, feedback_time TIMESTAMP)
        COMMENT 'Care-manager approve/reject decisions and free-text comments captured in the Offer Management Console. Ground truth for retraining.'""")

 ## Business metadata

In [ ]:
TABLE_COMMENTS = {
    "member_features": "80+ engineered member features (demographics, claims, diagnosis flags, benefits, engagement, temporal) used to train and score the offer model. Grain: one row per member, latest batch.",
    "offer_catalog": "Catalog of the 16 care and wellness programs the plan can offer (preventive, chronic disease, mental health, pharmacy, wellness, cost savings) with targeting rules.",
    "offer_recommendations": "Model output: top-5 next-best offers for every member with priority score (0-100), rank and model version. Grain: member x offer.",
    "member_offer_recommendations_with_reasoning": "Care-manager work queue: top-5 offers for sampled members with SHAP decision factors and LLM-written plain-language reasoning. Served to the app through Lakebase. PK (member_id, offer_id).",
    "model_feature_importance": "Mean LightGBM gain per feature across the 16 per-offer models for the current champion model version.",
}
for t, c in TABLE_COMMENTS.items():
    run(f"COMMENT ON TABLE {FQ}.{t} IS '{c}'")

COLUMN_COMMENTS = {
    "offer_recommendations": {
        "priority_score": "Model relevance score 0-100 for this member/offer; offers below 25 are not recommended.",
        "rank": "1 = most relevant offer for the member.",
        "model_version": "Unity Catalog model version of healthcare_offer_prioritizer that produced the score.",
    },
    "member_offer_recommendations_with_reasoning": {
        "llm_reasoning": "Plain-language explanation generated by databricks-gpt-5-1 from the member profile and SHAP factors.",
        "shap_factors": "JSON array of the top 5 features driving this score: feature, value, shap_value, direction.",
        "reasoning_source": "llm when generated by the Foundation Model; fallback when a templated sentence was used.",
    },
    "offer_feedback": {
        "feedback": "approved or rejected (button), or comment (free-text note), set by a care manager in the app.",
        "feedback_text": "Optional free-text comment from the care manager.",
    },
}
for t, cols in COLUMN_COMMENTS.items():
    for col, c in cols.items():
        run(f"ALTER TABLE {FQ}.{t} ALTER COLUMN {col} COMMENT '{c}'")

 ## Tags

In [ ]:
# Tag keys/values follow this metastore's governed tag policies (keys are case-sensitive):
#   Domain in {Healthcare, HR, Sales, Operations}; medallion_layer in {bronze, silver, gold};
#   data_classification in {u-nnpi, cui, pii}; phi in {true, health_plan_beneficiary_number, ...}
run(f"ALTER SCHEMA {FQ} SET TAGS ('Domain' = 'Healthcare')")

LAYERS = {
    "bronze": ["bronze_members", "bronze_claims", "bronze_benefits_utilization", "bronze_engagement_history"],
    "silver": ["silver_members", "silver_claims", "silver_benefits_utilization", "silver_engagement_history"],
    "gold": ["gold_member_360", "gold_claims_monthly", "gold_engagement_performance", "member_features",
             "offer_catalog", "offer_recommendations", "member_offer_recommendations_with_reasoning",
             "model_feature_importance", "offer_feedback"],
}
KIND = {t: k for k, ts in {
    "STREAMING TABLE": LAYERS["bronze"] + LAYERS["silver"],
    "MATERIALIZED VIEW": ["gold_member_360", "gold_claims_monthly", "gold_engagement_performance"],
}.items() for t in ts}
# Aggregates and reference data carry no member-level fields, so no data_classification
NOT_MEMBER_LEVEL = {"gold_claims_monthly", "gold_engagement_performance", "offer_catalog", "model_feature_importance"}
for layer, tables in LAYERS.items():
    for t in tables:
        kind = KIND.get(t, "TABLE")
        cls = "" if t in NOT_MEMBER_LEVEL else ", 'data_classification' = 'pii'"
        run(f"ALTER {kind} {FQ}.{t} SET TAGS ('medallion_layer' = '{layer}', 'Domain' = 'Healthcare'{cls})")

PHI_COLUMNS = {
    "member_features": ["member_id", "age", "risk_score"],
    "offer_recommendations": ["member_id"],
    "member_offer_recommendations_with_reasoning": ["member_id", "age", "risk_score", "has_diabetes",
                                                    "has_cardiovascular", "has_respiratory", "has_mental_health"],
    "offer_feedback": ["member_id", "feedback_text"],
}
# `phi` is a governed tag: member_id maps to the HIPAA identifier type
# health_plan_beneficiary_number; other member-level health attributes are phi=true.
for t, cols in PHI_COLUMNS.items():
    for col in cols:
        value = "health_plan_beneficiary_number" if col == "member_id" else "true"
        run(f"ALTER TABLE {FQ}.{t} ALTER COLUMN {col} SET TAGS ('phi' = '{value}')")

 ## Least-privilege grants

In [ ]:
run(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{APP_SP}`")
run(f"GRANT USE SCHEMA ON SCHEMA {FQ} TO `{APP_SP}`")
for t in ["member_offer_recommendations_with_reasoning", "offer_catalog", "offer_feedback"]:
    run(f"GRANT SELECT ON TABLE {FQ}.{t} TO `{APP_SP}`")
run(f"GRANT MODIFY ON TABLE {FQ}.offer_feedback TO `{APP_SP}`")

run(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{ANALYSTS}`")
run(f"GRANT USE SCHEMA ON SCHEMA {FQ} TO `{ANALYSTS}`")
for t in LAYERS["gold"]:
    run(f"GRANT SELECT ON TABLE {FQ}.{t} TO `{ANALYSTS}`")
# Registered models are FUNCTION securables in the UC grants API
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import PermissionsChange, Privilege
try:
    WorkspaceClient().grants.update(
        securable_type="function", full_name=f"{FQ}.healthcare_offer_prioritizer",
        changes=[PermissionsChange(principal=ANALYSTS, add=[Privilege.EXECUTE])])
    results.append(("OK", f"GRANT EXECUTE ON MODEL {FQ}.healthcare_offer_prioritizer TO `{ANALYSTS}` (grants API)"))
except Exception as e:
    results.append(("SKIP", f"GRANT EXECUTE ON MODEL {FQ}.healthcare_offer_prioritizer -- {str(e)[:140]}"))

for status, sql in results:
    print(f"[{status}] {' '.join(sql.split())[:220]}")
print(f"\n{sum(s == 'OK' for s, _ in results)} applied, {sum(s == 'SKIP' for s, _ in results)} skipped")

[OK] CREATE TABLE IF NOT EXISTS bhavik_demos.offer_prioritization.offer_feedback ( member_id STRING, offer_id STRING, feedback STRING, feedback_text STRING, feedback_time TIMESTAMP) COMMENT 'Care-manager approve/reject decisi
[OK] COMMENT ON TABLE bhavik_demos.offer_prioritization.member_features IS '80+ engineered member features (demographics, claims, diagnosis flags, benefits, engagement, temporal) used to train and score the offer model. Grain
[OK] COMMENT ON TABLE bhavik_demos.offer_prioritization.offer_catalog IS 'Catalog of the 16 care and wellness programs the plan can offer (preventive, chronic disease, mental health, pharmacy, wellness, cost savings) with tar
[OK] COMMENT ON TABLE bhavik_demos.offer_prioritization.offer_recommendations IS 'Model output: top-5 next-best offers for every member with priority score (0-100), rank and model version. Grain: member x offer.'
[OK] COMMENT ON TABLE bhavik_demos.offer_prioritization.member_offer_recommendations_with_reasoning IS 'Care-

 ## Audit: what is governed now

In [ ]:
display(spark.sql(f"""
  SELECT table_name, table_type, comment
  FROM {CATALOG}.information_schema.tables WHERE table_schema = '{SCHEMA}' ORDER BY table_name"""))

table_name | table_type | comment
--- | --- | ---
__materialization_mat_af3f71ac_5d82_492d_8f38_cd37284e7136_bronze_benefits_utilization_1 | MANAGED | Raw benefits_utilization extract (synthetic) ingested with Auto Loader from the raw_landing volume. One row per source record per landed batch.
__materialization_mat_af3f71ac_5d82_492d_8f38_cd37284e7136_bronze_claims_1 | MANAGED | Raw claims extract (synthetic) ingested with Auto Loader from the raw_landing volume. One row per source record per landed batch.
__materialization_mat_af3f71ac_5d82_492d_8f38_cd37284e7136_bronze_engagement_history_1 | MANAGED | Raw engagement_history extract (synthetic) ingested with Auto Loader from the raw_landing volume. One row per source record per landed batch.
__materialization_mat_af3f71ac_5d82_492d_8f38_cd37284e7136_bronze_members_1 | MANAGED | Raw members extract (synthetic) ingested with Auto Loader from the raw_landing volume. One row per source record per landed batch.
__materialization_mat_af3f71

In [ ]:
display(spark.sql(f"""
  SELECT table_name, tag_name, tag_value FROM {CATALOG}.information_schema.table_tags
  WHERE schema_name = '{SCHEMA}' ORDER BY table_name, tag_name"""))

table_name | tag_name | tag_value
--- | --- | ---
bronze_benefits_utilization | Domain | Healthcare
bronze_benefits_utilization | data_classification | pii
bronze_benefits_utilization | medallion_layer | bronze
bronze_claims | Domain | Healthcare
bronze_claims | data_classification | pii
bronze_claims | medallion_layer | bronze
bronze_engagement_history | Domain | Healthcare
bronze_engagement_history | data_classification | pii
bronze_engagement_history | medallion_layer | bronze
bronze_members | Domain | Healthcare
bronze_members | data_classification | pii
bronze_members | medallion_layer | bronze
gold_claims_monthly | Domain | Healthcare
gold_claims_monthly | medallion_layer | gold
gold_engagement_performance | Domain | Healthcare
gold_engagement_performance | medallion_layer | gold
gold_member_360 | Domain | Healthcare
gold_member_360 | data_classification | pii
gold_member_360 | medallion_layer | gold
member_features | Domain | Healthcare
member_features | data_classification | pi

In [ ]:
display(spark.sql(f"""
  SELECT table_name, column_name, tag_name, tag_value FROM {CATALOG}.information_schema.column_tags
  WHERE schema_name = '{SCHEMA}' ORDER BY table_name, column_name"""))

table_name | column_name | tag_name | tag_value
--- | --- | --- | ---
member_features | age | phi | true
member_features | member_id | phi | health_plan_beneficiary_number
member_features | risk_score | phi | true
member_offer_recommendations_with_reasoning | age | phi | true
member_offer_recommendations_with_reasoning | has_cardiovascular | phi | true
member_offer_recommendations_with_reasoning | has_diabetes | phi | true
member_offer_recommendations_with_reasoning | has_mental_health | phi | true
member_offer_recommendations_with_reasoning | has_respiratory | phi | true
member_offer_recommendations_with_reasoning | member_id | phi | health_plan_beneficiary_number
member_offer_recommendations_with_reasoning | risk_score | phi | true
offer_feedback | feedback_text | phi | true
offer_feedback | member_id | phi | health_plan_beneficiary_number
offer_recommendations | member_id | phi | health_plan_beneficiary_number

In [ ]:
display(spark.sql(f"""
  SELECT grantee, table_name, privilege_type FROM {CATALOG}.information_schema.table_privileges
  WHERE table_schema = '{SCHEMA}' AND grantee IN ('{APP_SP}', '{ANALYSTS}') ORDER BY grantee, table_name, privilege_type"""))

grantee | table_name | privilege_type
--- | --- | ---
account users | gold_claims_monthly | SELECT
account users | gold_engagement_performance | SELECT
account users | gold_member_360 | SELECT
account users | member_features | SELECT
account users | member_offer_recommendations_with_reasoning | SELECT
account users | model_feature_importance | SELECT
account users | offer_catalog | SELECT
account users | offer_feedback | SELECT
account users | offer_recommendations | SELECT
f9bec3ef-421b-4a07-bf08-46c6df20c976 | member_offer_recommendations_with_reasoning | SELECT
f9bec3ef-421b-4a07-bf08-46c6df20c976 | offer_catalog | SELECT
f9bec3ef-421b-4a07-bf08-46c6df20c976 | offer_feedback | MODIFY
f9bec3ef-421b-4a07-bf08-46c6df20c976 | offer_feedback | SELECT

In [ ]:
from mlflow.tracking import MlflowClient
import mlflow
mlflow.set_registry_uri("databricks-uc")
champion = MlflowClient().get_model_version_by_alias(f"{FQ}.healthcare_offer_prioritizer", "champion")
print(f"Model {FQ}.healthcare_offer_prioritizer: @champion -> v{champion.version} (run {champion.run_id})")

dbutils.notebook.exit(json.dumps({
    "applied": sum(s == "OK" for s, _ in results),
    "skipped": [sql for s, sql in results if s == "SKIP"],
}))

{"applied": 63, "skipped": []}